In [ ]:
<a href="https://colab.research.google.com/github/cia-group/tabforest/blob/main/run_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TNC Title + Abstract Relavance Prediction Tool

Welcome! This tool is designed to take a csv of agroforestry papers with title + abstract information and use a pre-trained natural language processing (NLP) model to predict whether each piece of text is 'Relevant' (1) or 'Irrelevant' (0). It will output a file with your input data along with new columns of the predictions made.


To get started, please **run the following cell to set up the environment**. It might take a few minutes. \
This colab will give a message that it crashed but you can ignore that. Your session will restart, after which you don't need to rerun this cell, just move on to the next steps.

In [ ]:
# This cell installs the required Python libraries.
print("Installing required libraries... This may take a few minutes.")
import os
%sx pip install -q --upgrade --force-reinstall \
    numpy==1.26.4 \
    typing-extensions --upgrade \
    tensorflow==2.12.1 \
    tensorflow-hub==0.13.0 \
    tensorflow-text==2.12.1
print("Libraries installed.")

os.kill(os.getpid(), 9)

## Step 0: Run Helper Functions

The following cells contain all the code that will run behind the scenes when you input your own data and model choice. \
All the functionality in this colab is packaged into functions so the code you work with later is easier to follow. Feel free to look through the helper functions, but there's no need to understand everything.

**TO-DO**: Run all the following cells. You can accomplish this by simply clicking the dropdown icon next to "Step 0" above to collapse this section, and then clicking the play button that appears at the bottom of this block.

In [1]:
# import libraries
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import torch
import torch.nn as nn
import torch.optim as optim
import requests
import warnings
import shutil
import sys
import os
pd.options.mode.copy_on_write = True

In [2]:
# function for retrieving models from github

GITHUB_URLS = {"BERT": "https://github.com/cia-group/tabforest/raw/main/wzheng/bert_precision.zip",
               "SPECTER-1LAYER": "https://github.com/cia-group/tabforest/raw/refs/heads/main/saved_models/specter-1layer-model.zip",
               "SPECTER-3LAYER": "https://github.com/cia-group/tabforest/raw/refs/heads/main/saved_models/specter-3layer-model.zip",
               "MINIBERT": "https://github.com/cia-group/tabforest/raw/refs/heads/main/saved_models/minibert_model.zip",
               "SCIBERT": "https://github.com/cia-group/tabforest/raw/refs/heads/main/saved_models/scibert_model.zip",
               "NAIVEBAYES": "https://github.com/cia-group/tabforest/raw/refs/heads/main/saved_models/naivebayes_model.zip",
               }

ZIP_FILENAMES = {"BERT": "bert_precision.zip",
                 "SPECTER-1LAYER": "specter-1layer-model.zip",
                 "SPECTER-3LAYER": "specter-3layer-model.zip",
                 "MINIBERT": "minibert_model.zip",
                 "SCIBERT": "scibert_model.zip",
                 "NAIVEBAYES": "naivebayes_model.zip",
                 }

MODEL_FOLDER_PATHS = {"SPECTER-1LAYER": "/content/specter-1layer.pt",
                      "SPECTER-3LAYER": "/content/specter-3layer.pt",
                      "BERT": "/content/bert_precision_hyperparam",
                      "MINIBERT": "/content/minibert_model",
                      "SCIBERT": "/content/scibert_model",
                      "NAIVEBAYES": "/content/naivebayes.pickle",
                      }

def retrieve_model_from_github(model_choice):
    github_url = GITHUB_URLS.get(model_choice)
    zip_filename = ZIP_FILENAMES.get(model_choice)
    download_path = f'/content/{zip_filename}'
    unzipped_dir = MODEL_FOLDER_PATHS.get(model_choice)

    if os.path.exists(unzipped_dir):
        print(f"Model path '{unzipped_dir}' already exists. Skipping download and unzip.")
    else:
        print(f"Downloading '{zip_filename}' from GitHub...")
        try:
            response = requests.get(github_url, stream=True)
            response.raise_for_status()
            with open(download_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Download complete.")

            print(f"Unzipping '{zip_filename}'...")
            shutil.unpack_archive(download_path, format='zip')
            print(f"Successfully unzipped to '{unzipped_dir}'")

            # remove the downloaded zip file after unzipping
            os.remove(download_path)
            print(f"Removed downloaded zip file: {zip_filename}")

        # handle errors
        except requests.exceptions.RequestException as e:
            print(f"ERROR during download: {e}")
            print(f"Could not download the file from '{github_url}'.")
        except FileNotFoundError:
            print(f"ERROR: Could not find the downloaded file '{download_path}' to unzip.")
        except shutil.ReadError:
            print(f"ERROR: Could not unzip the file '{zip_filename}'. It might be corrupted or not a valid zip file.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

In [ ]:
# helper functions for running predictions

from transformers import AutoModel
from transformers import AutoTokenizer
specter_model = AutoModel.from_pretrained('allenai/specter')
specter_model.eval() # set SPECTER tokenizer to inference mode inference mode
specter_tokenizer = AutoTokenizer.from_pretrained('allenai/specter')
def SPECTER_helper(example):
    """
    Helper function to embed a single example using SPECTER.
    input:: single entry from Hugging Face dataset
    return:: embed TAB column to some ex) [0.123, -0.456, 0.789, ..., 0.001]  # Shape: (768,)
    """
    input_text = example["TAB_preproc"]
    inputs = specter_tokenizer(input_text, return_tensors = "pt", truncation = True, max_length = 512) # Dict {'input_ids': tensor, 'token_type_ids': tensor}

    # import torch
    with torch.no_grad(): # disables gradient tracking...we are directly using SPECTER as an encoder as is, not training --> no need for backprop & store gradient
        outputs = specter_model(**inputs) # ** unpacks the dictionary
        cls_emb = outputs.last_hidden_state[:, 0, :]  # extract CLS token...[:, 0, :] all items, first item, all items
    return {"embedding": cls_emb.squeeze().numpy()}

from datasets import Dataset, Features, Sequence, Value
def encode_with_SPECTER(df):
    """
    Maps encode_with_SPECTER_helper to entire Hugging Face dataset.
    input:: Hugging Face dataset
    return:: Hugging Face dataset with new feature called 'embedding' w/ SPECTER embeddings, each with shape (768,)
    """
    # Change pandas dataframe to Hugging Face dataset
    HF_df = Dataset.from_pandas(df)

    # Add 'embedding' feature as a sequence of floats
    features = HF_df.features.copy()
    features["embedding"] = Sequence(Value("float32"))

    # Embed TAB
    tokenized_df = HF_df.map(
        SPECTER_helper,
        features = features,
        batched = False)
    print("Embedding complete.")
    return tokenized_df

from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader
def bert_build_dataloader(user_input):
    # built dataset
    user_dataset = Dataset.from_pandas(user_input["TAB_preproc"].to_frame())

    # tokenize data
    tokenizer = AutoTokenizer.from_pretrained("boltuix/bert-mini")
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    def tokenize_function(example):
        return tokenizer(example["TAB_preproc"], truncation=True, max_length=512)
    tokenized_dataset = user_dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.remove_columns(["TAB_preproc"])
    if '__index_level_0__' in tokenized_dataset.features:
        tokenized_dataset = tokenized_dataset.remove_columns(["__index_level_0__"])
    tokenized_dataset.set_format("torch")

    # build and return dataloader
    return DataLoader(tokenized_dataset, batch_size=64, collate_fn=data_collator)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight
# retraining naive bayes beacuse downloading the saved model leads to potential version errors
def retrain_naivebayes():
    # get re-training data
    if not(os.path.exists("/content/TAB_new.csv")):
        !wget -q https://github.com/cia-group/tabforest/raw/refs/heads/main/training_data/TAB_new.csv
    train_data = pd.read_csv("TAB_new.csv")

    # prepare data
    X_train, X_test, y_train, y_test = train_test_split(train_data['TAB_preproc'], train_data['label'], test_size=0.2, random_state=2025)
    # get TFIDFs
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),   # Include bigrams (e.g., "tree density")
        max_features=10000)
    # Fit on and transform training data
    X_train_tfidf = vectorizer.fit_transform(X_train)

    # init and train model
    model = MultinomialNB()
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    model.fit(X_train_tfidf, y_train, sample_weight=sample_weights)
    return model, vectorizer


In [4]:
# function for running predictions and summarizing results

PREDICTION_THRESHOLD = {"BERT": 0.8735,
                        "SPECTER-1LAYER": 0.5,
                        "SPECTER-3LAYER": 0.5,
                        "MINIBERT": 0.72,
                        "SCIBERT": 0.79,
                        }

prec_scores = {"BERT": 70,
               "SPECTER-1LAYER": 33,
               "SPECTER-3LAYER": 22,
               "MINIBERT": 53,
               "SCIBERT": 52,
               "NAIVEBAYES": 33,
               }

recall_scores = {"BERT": 23,
                "SPECTER-1LAYER": 77,
                "SPECTER-3LAYER": 96,
                "MINIBERT": 54,
                "SCIBERT": 55,
                "NAIVEBAYES": 81,
                }

from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification
def make_predictions(model_choice, user_input):
    warnings.filterwarnings("ignore")
    model_path = MODEL_FOLDER_PATHS.get(model_choice)
    if model_choice != "NAIVEBAYES":
        # NaiveBayes model will be retrained, not loaded
        print(f"Loading model from: {model_path}")

    # Predict with BERT ------------------------------------------
    if model_choice == "BERT":
        # load model
        model = tf.saved_model.load(model_path)

        # Make predictions
        print("Making predictions...")
        text_to_predict = tf.constant(user_input["TAB_preproc"].astype(str).tolist())
        infer = model.signatures["serving_default"]
        try:
            # raw_predictions = infer(text=text_to_predict)['classifier'].numpy() # Changed output key to 'classifier'
            # batching predictions to avoid memory overload
            batch_size = 64  # adjust based on your GPU/CPU memory
            raw_predictions = []
            progress_bar = tqdm(range(len(text_to_predict)//batch_size))
            for i in range(0, len(text_to_predict), batch_size):
                batch = text_to_predict[i:i+batch_size]
                preds = infer(text=batch)['classifier'].numpy()
                raw_predictions.append(preds)
                progress_bar.update(1)
            raw_predictions = np.concatenate(raw_predictions, axis=0)
        except KeyError as e:
            print(f"KeyError: {e}")
            print("Available output keys:")
            print(infer(text=text_to_predict).keys())
            sys.exit() # Exit after printing keys to avoid further errors

    # Predict with SPECTER -----------------------------------------
    elif model_choice == "SPECTER-1LAYER" or model_choice == "SPECTER-3LAYER":
        # load model
        model = torch.load(model_path, weights_only = False)
        model.eval()

        # encode input
        print("embedding TAB with SPECTER...")
        encoded = encode_with_SPECTER(user_input)
        input = torch.tensor(encoded["embedding"], dtype = torch.float32)

        # Make predictions
        print("Making predictions...")
        raw_predictions = model(input)
        raw_predictions = raw_predictions.detach().flatten().numpy()

    # Predict with MINIBERT or SCIBERT ------------------------------
    elif model_choice == "MINIBERT" or model_choice == "SCIBERT":
        user_dataloader = bert_build_dataloader(user_input)
        # load model
        device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        print(f"using device {device}. If you want to speed up runtime, change the runtime type to T4 GPU to use the 'cuda' device instead of the default 'cpu'.")
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
        # run predictions
        print("Making predictions...")
        progress_bar = tqdm(range(len(user_dataloader)))
        model.eval()
        all_probs = []
        for batch in user_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.no_grad():
                outputs = model(**batch)
            probs = torch.sigmoid(outputs.logits)
            all_probs.append(probs.detach().cpu().numpy())
            progress_bar.update(1)
        all_probs = [prob.item() for batch in all_probs for prob in batch]
        raw_predictions = np.asarray(all_probs)

    # Predict with NAIVEBAYES ------------------------------
    elif model_choice == "NAIVEBAYES":
        # retrain model (downloading saved model leads to potential version errors)
        model, vectorizer = retrain_naivebayes()
        # run predictions
        print("Making predictions...")
        user_data_tfidf = vectorizer.transform(user_input["TAB"])
        predictions = model.predict(user_data_tfidf)

    else:
        print("Error: The given model was not recognized. Please make sure your MODEL_CHOICE is valid and spelled correctly")

    # Add results to the dataframe -------------------------------
    if model_choice == "NAIVEBAYES":
        # NB does not return prediction scores, only predicted classes
        data["prediction"] = predictions
    else:
        data['prediction_score'] = raw_predictions#.detach().flatten().numpy() # raw probability from the model
        data['prediction'] = (data['prediction_score'] >= PREDICTION_THRESHOLD.get(model_choice)).astype(int)

    # print summary
    print("\n Prediction Complete. ")
    print("\nHere is a preview of your results:")
    display(data.head())

    print("\nSummary of Predictions:")
    print(data['prediction'].astype(str).value_counts())

    pred0, pred1 = data['prediction'].value_counts().values
    print("")
    print("Of the papers that were predicted as relevant,")
    print(f"- About {prec_scores[model_choice]}% of these should be truly relevant")
    print(f"- About {recall_scores[model_choice]}% of all the truly relevant papers should be present \n")
    warnings.filterwarnings("default")

In [ ]:
# functions for dealing with data reading/downloading

def read_data(UPLOADED_CSV_PATH):
    try:
        df = pd.read_csv(UPLOADED_CSV_PATH)
        # Ensure the given columns exist
        if TITLE_COLUMN_NAME not in df.columns:
            print(f"ERROR: Column '{TITLE_COLUMN_NAME}' not found in your CSV file.")
            print(f"Available columns are: {list(df.columns)}")
        elif ABS_COLUMN_NAME not in df.columns:
            print(f"ERROR: Column '{ABS_COLUMN_NAME}' not found in your CSV file.")
            print(f"Available columns are: {list(df.columns)}")
    except FileNotFoundError:
        print(f"ERROR: Could not find the CSV file at '{UPLOADED_CSV_PATH}'.")
        print("Please make sure you have uploaded the file and provided the proper name.")
    return df

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
def preprocess_data(df):
    # combine title and abstract
    df["TAB"] = np.where(df[ABS_COLUMN_NAME].notna(), df[TITLE_COLUMN_NAME] + " " + df[ABS_COLUMN_NAME], df[TITLE_COLUMN_NAME])
    # preprocess text: convert to lowercase, remove stopwords and punctuation
    stop_words = set(stopwords.words("english"))
    def tokenize(text):
        tokens = [
            token.lower()   # convert to lowercase
            for token in text.split()   # split text by whitespace
            if token.lower() not in stop_words and token.isalpha()  # filter out stopwords and non-alphabetic characters
        ]
        return ' '.join(tokens)
    df['TAB_preproc'] = df["TAB"].apply(tokenize)
    return df

from google.colab import files
def save_predictions(output_filename):
    warnings.filterwarnings("ignore")
    print(f"Saving results to {output_filename}...")
    data.to_csv(output_filename, index=False)

    print(f"Downloading results to your computer...")
    files.download(output_filename)
    warnings.filterwarnings("default")

## Step 1: Upload Your Files

Here you will upload your csv data file containing the titles and abstracts of papers you want to run relevance predictions on.

**TO-DO:**
* On the left sidebar of this Colab window, click the folder icon to open the File Browser.
* Click the 'Upload to session storage' button (the icon of a page with an upward arrow).
* Select the `.csv` file from your computer that contains your title + abstract data.

Once uploaded, you should see your CSV file in the file browser.

## Step 2: Set Your Parameters

Here you will specify where to find your data and which model you want to use.

**TO-DO:** Carefully edit the following variables in the code cell below, then press run.
1. `UPLOADED_CSV_PATH`: The name of the data file you uploaded.
2. `TITLE_COLUMN_NAME` + `ABS_COLUMN_NAME`: The column names for the titles and abstracts in your csv file.
3. `MODEL_CHOICE`: Which model you want to use to make predictions. See the table below for information on the available models. Refer to the GitHub README for precision and recall definitions.
4. `OUTPUT_FILENAME`: What you would like the resulting output file to be named

**Note:** all of your input values should be in quotation marks.

| Model Name    | Precision | Recall | Notes  |
| :-------------: | :---------: | :------: | :------ |
| BERT     |  0.70     |  0.23  |  **highest precision** (read through least amount of irrelevant papers but capture less relevant papers from input)      |
| MINIBERT     |  0.53     |  0.54  | of the bert models, the smallest and quickest to run      |
| SCIBERT       |  0.52     |  0.55  |  largest model (takes most time/compute power to run)      |
| SPECTER-1LAYER     |  0.33     |  0.77  |  of the specter models, the smallest and quickest to run      |
| SPECTER-3LAYER     |  0.22     |  0.96  |  **highest recall** (capture more relevant papers from input but sort through more irrelevant papers)      |
| NAIVEBAYES    |  0.33     |  0.81  |  smallest model (takes least time/compute power to run)      |

In [6]:
# 1. YOUR UPLOADED CSV FILE
UPLOADED_CSV_PATH = "your_data.csv"

# 2. TITLE + ABSTRACT COLUMN NAMES
TITLE_COLUMN_NAME = "title"
ABS_COLUMN_NAME = "abstract"

# 3. MODEL CHOICE
# Options: "BERT", "SCIBERT", "MINIBERT", "SPECTER-1LAYER", "SPECTER-3LAYER", "NAIVEBAYES"
MODEL_CHOICE = "BERT"

# 4. OUTPUT FILENAME
OUTPUT_FILENAME = 'predictions_output.csv'

## Step 3: Run the Prediction!

Now you are ready to run the model. The code below will handle everything automatically based on your settings from Step 2. It will read in and prepare your data, retrieve your chosen model, run predictions, and download the results to your computer.

**TO-DO:** Run the following cell and relax until the results are downloaded for you.

In [ ]:
# read in your csv file
data = read_data(UPLOADED_CSV_PATH)

# prepare your data
input = preprocess_data(data)

# retrieve your chosen model
retrieve_model_from_github(MODEL_CHOICE)

# run your predictions
make_predictions(MODEL_CHOICE, input)

# save your predictions
save_predictions(OUTPUT_FILENAME)